<a href="https://colab.research.google.com/github/AbderrahmenHachicha/CardioDiffusion/blob/main/notebooks/03_data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
df_ecg_gen = pd.read_csv('/content/drive/MyDrive/Projet PFA/1_Data/generated/ecg_generated.csv')
df_ecg_pro = pd.read_csv('/content/drive/MyDrive/Projet PFA/1_Data/processed/ecg_segments.csv')


In [4]:
print(df_ecg_gen.shape)
print(df_ecg_pro.shape)

(13000, 188)
(111328, 189)


In [5]:
df_ecg_pro = df_ecg_pro.drop("patient", axis=1)

In [7]:
df_final = pd.concat([df_ecg_pro, df_ecg_gen], ignore_index=True)
keep = ['N', 'A', 'V', 'L', 'R']
df_final = df_final[df_final['label'].isin(keep)]
print(df_final.shape)
print(df_final['label'].value_counts())

(113038, 188)
label
N    76033
L    11073
R    10257
V    10129
A     5546
Name: count, dtype: int64


In [8]:
label_map={"N" : 0 , "A" : 1,"V" : 2 ,"L" : 3, "R":4 }
df_final['label'] = df_final['label'].map(label_map)
print(df_final['label'].value_counts())
def normalize(df):
    df_signals = df.iloc[:, :187]
    df_norm = df_signals.apply(lambda row: (row - row.min()) / (row.max() - row.min()), axis=1)
    return df_norm

label
0    76033
3    11073
4    10257
2    10129
1     5546
Name: count, dtype: int64


In [9]:
df_norm = normalize(df_final)
print(df_norm.min().min())
print(df_norm.max().max())

0.0
1.0


In [10]:
from sklearn.model_selection import train_test_split

In [11]:
df_combined = df_norm.copy()
df_combined['label'] = df_final['label'].values
train_df, temp_df = train_test_split(
    df_combined,
    test_size=0.20,
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42
)
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (90430, 188)
Validation: (11304, 188)
Test: (11304, 188)


In [12]:
train_df.to_csv('/content/drive/MyDrive/Projet PFA/1_Data/processed/ecg_final_train.csv', index=False)
val_df.to_csv('/content/drive/MyDrive/Projet PFA/1_Data/processed/ecg_final_val.csv', index=False)
test_df.to_csv('/content/drive/MyDrive/Projet PFA/1_Data/processed/ecg_final_test.csv', index=False)
print("All splits saved!")

✅ All splits saved!
